**数据预处理**
:label:sec_pandas

为了能用深度学习来解决现实世界的问题，我们经常从预处理原始数据开始，
而不是从那些准备好的张量格式数据开始。
在Python中常用的数据分析工具中，我们通常使用pandas软件包。
像庞大的Python生态系统中的许多其他扩展包一样，pandas可以与张量兼容。
本节我们将简要介绍使用pandas预处理原始数据，并将原始数据转换为张量格式的步骤。
后面的章节将介绍更多的数据预处理技术。

**读取数据集**
举一个例子，我们首先(创建一个人工数据集，并存储在CSV（逗号分隔值）文件)
../data/house_tiny.csv中。
以其他格式存储的数据也可以通过类似的方式进行处理。
下面我们将数据集按行写入CSV文件中。

In [1]:
import os  # 导入 os 模块，用来处理路径、创建文件夹等

# os.path.join('..', 'data') 会拼出一个相对路径：../data
# '..' 表示“当前文件夹的上一级文件夹”
# 所以这里的意思是：在上一级目录里创建一个 data 文件夹
# exist_ok=True 表示如果 data 文件夹已经存在，就不要报错
os.makedirs(os.path.join('..', 'data'), exist_ok=True)

# 再拼出完整的数据文件路径：../data/house_tiny.csv
# 这个变量 data_file 以后就代表这个 csv 文件的位置
data_file = os.path.join('..', 'data', 'house_tiny.csv')

# with open(..., 'w') 表示“以写入模式打开文件”
# 如果文件不存在，会自动创建
# 如果文件已经存在，会覆盖原来的内容
# as f 表示把这个打开的文件对象命名为 f
with open(data_file, 'w') as f:
    # 第一行写入表头，也就是每一列的名字
    # \n 表示换行，写完这一行后会自动到下一行
    f.write('NumRooms,Alley,Price\n')  # 列名：房间数、小巷类型、价格
    

    # 从这里开始，每一行都是一条样本数据
    # 逗号分隔的 3 个值，分别对应上面的 3 列
    # NA 表示这个位置的数据缺失了
    f.write('NA,Pave,127500\n')  # 房间数缺失，小巷类型是 Pave，价格是 127500
    f.write('2,NA,106000\n')     # 房间数是 2，小巷类型缺失，价格是 106000
    f.write('4,NA,178100\n')     # 房间数是 4，小巷类型缺失，价格是 178100
    f.write('NA,NA,140000\n')    # 房间数和小巷类型都缺失，价格是 140000

要[**从创建的CSV文件中加载原始数据集**]，我们导入`pandas`包并调用`read_csv`函数。该数据集有四行三列。其中每行描述了房间数量（“NumRooms”）、巷子类型（“Alley”）和房屋价格（“Price”）。

In [2]:
import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


## 处理缺失值

注意，“NaN”项代表缺失值。
[**为了处理缺失的数据，典型的方法包括*插值法*和*删除法*，**]
其中插值法用一个替代值弥补缺失值，而删除法则直接忽略缺失值。
在(**这里，我们将考虑插值法**)。

通过位置索引`iloc`，我们将`data`分成`inputs`和`outputs`，
其中前者为`data`的前两列，而后者为`data`的最后一列。
对于`inputs`中缺少的数值，我们用同一列的均值替换“NaN”项。


In [3]:
inputs = data.iloc[:,0:2]#inputs 是data里面所有行的前两列
outputs = data.iloc[:,2:3]#outputs 是data里面所有的第三列
print('inpts are\n',inputs)
print('outputs are\n',outputs)

inpts are
    NumRooms Alley
0       NaN  Pave
1       2.0   NaN
2       4.0   NaN
3       NaN   NaN
outputs are
     Price
0  127500
1  106000
2  178100
3  140000


[对于inputs中的类别值或离散值，我们将“NaN”视为一个类别。] 由于“巷子类型”（“Alley”）列只接受两种类型的类别值“Pave”和“NaN”， pandas可以自动将此列转换为两列“Alley_Pave”和“Alley_nan”。 巷子类型为“Pave”的行会将“Alley_Pave”的值设置为1，“Alley_nan”的值设置为0。 缺少巷子类型的行会将“Alley_Pave”和“Alley_nan”分别设置为0和1。

In [4]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       NaN        True      False
1       2.0       False       True
2       4.0       False       True
3       NaN       False       True


## 转换为张量格式
[**现在`inputs`和`outputs`中的所有条目都是数值类型，它们可以转换为张量格式。**]
当数据采用张量格式后，可以通过在 :numref:`sec_ndarray`中引入的那些张量函数来进一步操作。


In [5]:
import torch
X = torch.tensor(inputs.to_numpy(dtype=float))
Y = torch.tensor(outputs.to_numpy(dtype=float))
X,Y

OSError: [WinError 1114] 动态链接库(DLL)初始化例程失败。 Error loading "C:\Users\A\miniconda3\envs\d2l-py311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 练习
创建包含更多行和列的原始数据集。

1.删除缺失值最多的列。
2.将预处理后的数据集转换为张量格式。

In [ ]:
import os
import pandas as pd


os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny_5x5.csv')

with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Floor,Age,Price\n')   # 5 列
    f.write('NA,Pave,2,10,127500\n')             # 第 1 行
    f.write('2,NA,NA,15,106000\n')               # 第 2 行
    f.write('4,NA,3,NA,178100\n')                # 第 3 行
    f.write('NA,NA,NA,20,140000\n')              # 第 4 行
    f.write('3,Pave,1,8,150000\n')               # 第 5 行

data = pd.read_csv(data_file)
print(data)

In [ ]:
print(data.isna().sum())

In [ ]:
index = data.isna().sum().idxmax()
print(index)

In [ ]:
print(data.columns)

In [ ]:
data = pd.read_csv(data_file)   # 每次先回到原始数据
index = data.isna().sum().idxmax()
data = data.drop(columns=[index])
print(data)